In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder


print("Environment stable.")

/Users/siri/Desktop/sghpolicyreview/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment stable.


## **1. Load the JCI ESS document**

In [2]:
JCI_PDF = "/Users/siri/Desktop/sghpolicyreview/data/ESS Standards_7th Ed .pdf"

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(JCI_PDF)
pages = loader.load()

print(f"Total pages loaded = {len(pages)}")


Total pages loaded = 406


## **2. Grouping pages by chapter**

In [4]:
import re
from collections import defaultdict

VALID_CODES = {
    "APR","IPSG","ACC","PCC","AOP","COP","ASC","MMU",
    "QPS","PCI","GLD","FMS","SQE","MOI",
    "MPE","HRP"
}

def detect_chapter_from_text(text: str):
    lines = text.split("\n")
    for line in lines:
        line = line.strip()
        match = re.match(r'^.+\(([A-Z]{3,4})\)\s*$', line)
        if match:
            code = match.group(1)               
            if code in VALID_CODES:
                return code
    return None

# ── STEP 1: Build chapter_pages first ────────────────────────────────
chapter_pages = defaultdict(list)
current_chapter = "UNKNOWN"

for page in pages:
    detected = detect_chapter_from_text(page.page_content)
    if detected:
        current_chapter = detected
    chapter_pages[current_chapter].append(page)


# ── STEP 2: NOW clean end-of-book pages from HRP ─────────────────────
# The glossary/appendix pages at the end of the PDF get incorrectly 
# assigned to HRP because there's no new chapter title after it

END_OF_BOOK_MARKERS = ["glossary", "index", "appendix", "references"]

def is_end_of_book_page(page_text):
    first_200 = page_text.strip()[:200].lower()
    return any(marker in first_200 for marker in END_OF_BOOK_MARKERS)

if "HRP" in chapter_pages:
    cleaned_hrp = []
    removed = 0
    for page in chapter_pages["HRP"]:
        if is_end_of_book_page(page.page_content):
            removed += 1
        else:
            cleaned_hrp.append(page)
    chapter_pages["HRP"] = cleaned_hrp
    # print(f"\nRemoved {removed} end-of-book pages from HRP")
    # print(f"HRP now has {len(chapter_pages['HRP'])} pages")

    print("=== CHAPTER PAGE COUNTS ===")

total = 0
for chapter, pg in chapter_pages.items():
    print(f"  {chapter}: {len(pg)} pages")
    total += len(pg)

print(f"\nTotal pages: {total}")
# print(f"HRP after cleanup: {len(chapter_pages['HRP'])} pages (glossary removed)")

=== CHAPTER PAGE COUNTS ===
  UNKNOWN: 2 pages
  FMS: 42 pages
  APR: 18 pages
  ACC: 22 pages
  PCC: 21 pages
  AOP: 30 pages
  COP: 32 pages
  ASC: 16 pages
  QPS: 15 pages
  PCI: 27 pages
  GLD: 32 pages
  SQE: 28 pages
  MOI: 25 pages
  MPE: 13 pages
  HRP: 32 pages
  IPSG: 16 pages
  MMU: 22 pages

Total pages: 393


In [5]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def build_structured_chapter_chunks(chapter_pages, chunk_size=800, chunk_overlap=150):
    
    inner_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " "]
    )

    chapter_chunks = {}

    for chapter, pg_list in chapter_pages.items():
        
        # 1. Merge all pages into one string
        full_text = "\n".join([p.page_content for p in pg_list])

        # 2. Clean page artifacts
        full_text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', full_text) # clears the page no.s
        full_text = re.sub(r'\n{3,}', '\n\n', full_text) # clears the unecesssary white spaces

        # 3. Skip the TOC/summary section at the top
        # The real content starts at "Standards, Intents, and Measurable Elements"
        # which appears as a heading before the first real standard
        toc_marker = re.search(
            r'Standards,\s+Intents,\s+and\s+Measurable\s+Elements', # finds "Standards", then one or more whitespaces. then finds "Intents", then one or more whitespaces and so on.
            full_text,
            re.IGNORECASE # ignores upper and lower cases
        )
        if toc_marker:
            full_text = full_text[toc_marker.start():] # gives the character from the text where the match begins. deletes everyhting before this character and saves the content from that point onwards.
        
        # 4. Split by standard codes that BELONG to this chapter only
        # pattern = rf'(?:\n|^)Standard\s+({chapter}\.\d+(?:\.\d+)?)\b'  

        # start of the line/text, "Standard" followed by one/more whitespace, chapter code, one or more digits, word boundary-stops matching mid word.
        # \s* means zero or more spaces between chapter code and the dot
        # this handles both "ACC.1" and "AOP .1"
        # pattern = rf'(?:\n|^)Standard\s+({chapter}\s*\.\s*\d+(?:\s*\.\s*\d+)?)\b'

        # NEW — matches "Standard ACC.1", "Standard AOP .1", and "Requirement APR.1"
        pattern = rf'(?:\n|^)(?:Standard|Requirement)\s+({chapter}\s*\.\s*\d+(?:\s*\.\s*\d+)?)\b'
        splits = re.split(pattern, full_text) # this gives alternate code and content splits, eg: splits[0]=intro content before standards, splits[1]=code, splits[2]=content

        docs = []

        # splits[0] is intro text before any standard code — keep it
        if splits[0].strip():
            docs.append(Document(
                page_content=splits[0].strip(),
                metadata={"chapter": chapter, "standard_code": "INTRO"}
            ))

        for i in range(1, len(splits), 2):  #stepping through by 2 as the code and content alternates. 
            if i + 1 >= len(splits): #This guard prevents crashing if the list ends on a code with no content after it.
                break

            # standard_code = splits[i].strip()
            standard_code = splits[i].strip().replace(" ", "")
            content = f"{standard_code}\n{splits[i+1]}"

            if len(content.strip()) < 100: # if character size is. < 100 then skip as it cannot be a whole standard can be for eg: "See ACC.5.1 also."
                continue

            if len(content) <= chunk_size: # If the standard's content fits within chunk_size characters then it is kept as one single chunk. Short standards like ACC.4 stay intact.
                docs.append(Document(
                    page_content=content.strip(),
                    metadata={
                        "chapter": chapter,
                        "standard_code": standard_code
                    }
                ))
            else:
                sub_chunks = inner_splitter.split_text(content)
                for j, chunk_text in enumerate(sub_chunks):
                    if len(chunk_text.strip()) < 100:
                        continue
                    docs.append(Document(
                        page_content=chunk_text.strip(),
                        metadata={
                            "chapter": chapter,
                            "standard_code": standard_code,
                            "chunk_index": j # sub chunks given an index
                        }
                    ))

        chapter_chunks[chapter] = docs
        print(f"{chapter}: {len(docs)} chunks across "
              f"{len(set(d.metadata['standard_code'] for d in docs))} standards")

    return chapter_chunks


chapter_chunks = build_structured_chapter_chunks(
    chapter_pages,
    chunk_size=1000,
    chunk_overlap=250
)

UNKNOWN: 1 chunks across 1 standards
FMS: 131 chunks across 26 standards
APR: 32 chunks across 12 standards
ACC: 80 chunks across 17 standards
PCC: 79 chunks across 22 standards
AOP: 114 chunks across 31 standards
COP: 129 chunks across 29 standards
ASC: 50 chunks across 16 standards
QPS: 56 chunks across 12 standards
PCI: 106 chunks across 22 standards
GLD: 142 chunks across 34 standards
SQE: 128 chunks across 25 standards
MOI: 95 chunks across 17 standards
MPE: 23 chunks across 8 standards
HRP: 140 chunks across 10 standards
IPSG: 76 chunks across 14 standards
MMU: 85 chunks across 18 standards


# 7 Gold Standards

In [7]:
from collections import Counter

# gold_standard = [
#     # ACC.1 — Screening for admission
#     {"query": "How should hospitals evaluate whether a patient's needs match available services?", "expected_standard": "ACC.1", "expected_chapter": "ACC"},
#     {"query": "What are the requirements for triaging patients with urgent or emergency needs?", "expected_standard": "ACC.1", "expected_chapter": "ACC"},
#     # ACC.1.1 — Delays in service
#     {"query": "What must hospitals do when there are delays in providing care to patients?", "expected_standard": "ACC.1.1", "expected_chapter": "ACC"},
#     # ACC.2 — Patient flow
#     {"query": "How should hospitals manage patient flow and bed allocation throughout the facility?", "expected_standard": "ACC.2", "expected_chapter": "ACC"},
#     # ACC.2.3 — Admission and discharge criteria
#     {"query": "What criteria should be used to admit or discharge patients from specialised units?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},
#     # ACC.3 — Continuity of care
#     {"query": "How should information be shared between healthcare practitioners to ensure continuity?", "expected_standard": "ACC.3", "expected_chapter": "ACC"},
#     {"query": "What processes ensure care coordination when a patient moves between departments?", "expected_standard": "ACC.3", "expected_chapter": "ACC"},
#     # ACC.4 — Discharge planning
#     {"query": "When should discharge planning begin and what should it include?", "expected_standard": "ACC.4", "expected_chapter": "ACC"},
#     # ACC.4.1 — Patient education on discharge
#     {"query": "What instructions and information must be given to patients when they leave the hospital?", "expected_standard": "ACC.4.1", "expected_chapter": "ACC"},
#     # ACC.5 — Transfer
#     {"query": "What are the requirements for safely transferring a patient to another healthcare facility?", "expected_standard": "ACC.5", "expected_chapter": "ACC"},
#     {"query": "Who is responsible for monitoring a patient during transfer and what qualifications are needed?", "expected_standard": "ACC.5", "expected_chapter": "ACC"},
#     # ACC.5.1 — Transfer documentation
#     {"query": "What clinical information must be documented and handed over when transferring a patient?", "expected_standard": "ACC.5.1", "expected_chapter": "ACC"},
#     # ACC.6 — Transportation
#     {"query": "What safety and regulatory standards must hospital transportation services meet?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},
#     {"query": "How should hospitals assess and arrange safe transport for patients being discharged?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},
#     {"query": "What equipment and medications must be available in patient transport vehicles?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},
# ]

gold_standard = [

    # ACC.1 — Screening and Admission
    {"query": "What screening criteria should be used when a patient arrives at the emergency department?", "expected_standard": "ACC.1", "expected_chapter": "ACC"},
    {"query": "How should hospitals handle patients with communicable diseases during screening?", "expected_standard": "ACC.1", "expected_chapter": "ACC"},

    # ACC.2.2 — Admission information
    {"query": "What information must be provided to patients when they are admitted to the hospital?", "expected_standard": "ACC.2.2", "expected_chapter": "ACC"},

    # ACC.2 — Patient Flow and Bed Management
    {"query": "How should hospitals manage patient flow to prevent emergency department overcrowding?", "expected_standard": "ACC.2", "expected_chapter": "ACC"},
    {"query": "What are the requirements for managing patients waiting for inpatient beds in the ED?", "expected_standard": "ACC.2", "expected_chapter": "ACC"},

    # ACC.2.3 — Intensive Care and Specialized Services
    {"query": "What criteria should be established for admission to intensive care units?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},
    {"query": "How should hospitals determine when patients no longer need intensive care?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},

    # ACC.3.1 — Continuity of Care
    {"query": "Who should be responsible for coordinating a patient's care throughout their hospital stay?", "expected_standard": "ACC.3.1", "expected_chapter": "ACC"},
    {"query": "What processes should be in place to ensure continuity when care responsibility is transferred between providers?", "expected_standard": "ACC.3.1", "expected_chapter": "ACC"},

    # ACC.4 — Discharge Planning
    {"query": "When should discharge planning begin for hospitalized patients?", "expected_standard": "ACC.4", "expected_chapter": "ACC"},

    # ACC.4.1 — Discharge Education
    {"query": "What education must be provided to patients and families before discharge?", "expected_standard": "ACC.4.1", "expected_chapter": "ACC"},

    # ACC.4.2 — Discharge Summary
    {"query": "What information must be included in a patient's discharge summary?", "expected_standard": "ACC.4.2", "expected_chapter": "ACC"},

    # ACC.4.2.1 — Emergency Care Documentation
    {"query": "What specific information must be documented for emergency department patients?", "expected_standard": "ACC.4.2.1", "expected_chapter": "ACC"},

    # ACC.4.3 — Complex Care Patients
    {"query": "How should hospitals manage information for patients with multiple complex diagnoses?", "expected_standard": "ACC.4.3", "expected_chapter": "ACC"},

    # ACC.4.4 — AMA (Against Medical Advice)
    {"query": "What procedures should be followed when a patient wants to leave against medical advice?", "expected_standard": "ACC.4.4", "expected_chapter": "ACC"},
    {"query": "What documentation is required when a patient leaves without notifying staff?", "expected_standard": "ACC.4.4.1", "expected_chapter": "ACC"},

    # ACC.5 — Patient Transfer
    {"query": "What criteria should be used to determine if a patient needs to be transferred to another facility?", "expected_standard": "ACC.5", "expected_chapter": "ACC"},

    # ACC.5.1 — Transfer Documentation
    {"query": "What information must be provided to the receiving hospital when transferring a patient?", "expected_standard": "ACC.5.1", "expected_chapter": "ACC"},

    # ACC.6 — Transportation Services
    {"query": "What are the hospital's responsibilities for patient transportation services?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},

    # ACC.1.1 — Delays in Care
    {"query": "How should hospitals inform patients about delays in diagnostic or treatment services?", "expected_standard": "ACC.1.1", "expected_chapter": "ACC"},

    # Cross-standard queries (multi-label but mapped to primary)
    {"query": "What are all the documentation requirements when a patient is transferred from the ICU to another hospital?", "expected_standard": "ACC.5.1", "expected_chapter": "ACC"},
    {"query": "What education and follow-up is required for a complex care patient being discharged home?", "expected_standard": "ACC.4.1", "expected_chapter": "ACC"},

    # Edge cases
    {"query": "What are the requirements for psychiatric patient admission criteria?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},
    {"query": "How should hospitals handle patients requiring organ transplant services?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},
    {"query": "What infection control measures should be considered during patient transportation?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},

]
print(f"Gold standard: {len(gold_standard)} queries")
standard_counts = Counter(item["expected_standard"] for item in gold_standard)
print("\nQueries per standard:")
for standard, count in sorted(standard_counts.items()):
    print(f"  {standard}: {count} queries")

Gold standard: 25 queries

Queries per standard:
  ACC.1: 2 queries
  ACC.1.1: 1 queries
  ACC.2: 2 queries
  ACC.2.2: 1 queries
  ACC.2.3: 4 queries
  ACC.3.1: 2 queries
  ACC.4: 1 queries
  ACC.4.1: 2 queries
  ACC.4.2: 1 queries
  ACC.4.2.1: 1 queries
  ACC.4.3: 1 queries
  ACC.4.4: 1 queries
  ACC.4.4.1: 1 queries
  ACC.5: 1 queries
  ACC.5.1: 2 queries
  ACC.6: 2 queries


# **Evaluation experimental pipeline**

In [8]:
# ============================================================
# 🧪 EVALUATION PIPELINE (ACTIVE EXPERIMENTAL SETUP)
# ============================================================
# This section compares multiple embedding models by:
#
# 1. Rebuilding FAISS vector DBs for EACH model
# 2. Running a set of gold-standard queries
# 3. Measuring retrieval performance (Hit@K)
#
# Key behavior:
# - Each model creates its OWN vector space
# - DB is rebuilt from scratch for fair comparison
# - Query and chunks use the SAME embedding model
#
# IMPORTANT:
# - Does NOT use the saved FAISS DB from production pipeline
# - This is intentional for benchmarking model performance
#
# Goal:
# - Identify best embedding model for retrieval quality
# ============================================================

# Models to compare
# We load one at a time during evaluation to avoid memory issues
# MODELS_TO_EVALUATE = {
#     "MiniLM (baseline)" : "sentence-transformers/all-MiniLM-L6-v2",
#     "BGE-small"         : "BAAI/bge-small-en-v1.5",
#     "BGE-large"         : "BAAI/bge-large-en-v1.5",
# }


# k values to test
# hit@3 = did correct standard appear in top 3 results?
K_VALUES = [1, 3, 5]

# print("Models to evaluate:")
# for name, model_str in MODELS_TO_EVALUATE.items():
#     print(f"  {name}: {model_str}")

# -----------Testing only with BGE-large-------------------------------

emb_model = "BAAI/bge-large-en-v1.5"

print(f"\n[EMBEDDING] Loading: {emb_model}")

embeddings = HuggingFaceEmbeddings(
    model_name=emb_model,
    encode_kwargs={"normalize_embeddings": True}
)


# ---------DB Builder--------------------------------------------
def build_dbs(chapter_chunks, embeddings):
    print("Building DBs for all chapters")
    
    dbs = {}
    for chapter, docs in chapter_chunks.items():
        if docs:
            dbs[chapter] = FAISS.from_documents(docs, embeddings)
            print(f"Built {chapter} DB")
    
    return dbs


# ---------Building the DBs--------------------------------------------
print("\nBuilding DBs...")
dbs = build_dbs(chapter_chunks, embeddings)

print("\nAll DBs ready!")


[EMBEDDING] Loading: BAAI/bge-large-en-v1.5


/var/folders/r6/6d8w4h7s47x81h2h03y77ys40000gn/T/ipykernel_21294/894171591.py:46: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 925.93it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Building DBs...
Building DBs for all chapters
Built UNKNOWN DB
Built FMS DB
Built APR DB
Built ACC DB
Built PCC DB
Built AOP DB
Built COP DB
Built ASC DB
Built QPS DB
Built PCI DB
Built GLD DB
Built SQE DB
Built MOI DB
Built MPE DB
Built HRP DB
Built IPSG DB
Built MMU DB

All DBs ready!


# Retrieval function

In [9]:
# ================================
# RETRIEVAL 
# ================================
def retrieve_all(query, dbs, per_db_k=8, final_k=10):

    all_retrieved = []

    # ----------------------------
    # 1. Retrieve from each chapter DB
    # ----------------------------
    for chapter_name, db in dbs.items():
        res = db.similarity_search_with_score(query, k=per_db_k)

        for doc, score in res:
            doc.metadata["chapter"] = chapter_name
            all_retrieved.append((doc, score))

    # ----------------------------
    # 2. Global sort (IMPORTANT)
    # ----------------------------
    # Lower score = better (FAISS)
    all_retrieved = sorted(all_retrieved, key=lambda x: x[1])

    # ----------------------------
    # 3. Deduplication (IMPROVED)
    # ----------------------------
    seen = set()
    unique_docs = []

    for doc, score in all_retrieved:
        key = (doc.page_content, doc.metadata.get("standard_code"))

        if key not in seen:
            unique_docs.append((doc, score))
            seen.add(key)

    # ----------------------------
    # 4. Final cut
    # ----------------------------
    return unique_docs[:final_k]

# Re Ranker functions

In [10]:
# ================================
# CROSS ENCODER RERANK
# ================================
def rerank_cross(query, retrieved, model):

    docs = [doc for doc, _ in retrieved]

    pairs = [(query, d.page_content) for d in docs]
    scores = model.predict(pairs)

    return sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)


# ================================
# QWEN LOAD MODEL
# ================================
def load_qwen():
    from transformers import AutoTokenizer, AutoModelForCausalLM

    model_id = "Qwen/Qwen3-Reranker-4B"

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        device_map="auto"  # HPC friendly
    )

    model.eval()
    return tokenizer, model


def rerank_qwen(query, retrieved, tokenizer, model):

    import torch
    import re

    scored = []

    for doc, retr_score in retrieved:

        prompt = f"""You are a strict scoring function.

        Score relevance from 0 to 10.

        0 = irrelevant  
        5 = partially relevant  
        10 = directly answers the query  

        Examples:

        Query: When should discharge planning begin?
        Document: Discharge planning must begin at admission.
        Score: 10

        Query: When should discharge planning begin?
        Document: Hospitals must ensure coordination across departments.
        Score: 3

        Now evaluate:

        Query:
        {query}

        Document:
        {doc.page_content}

        Output ONLY a single number (0–10).
        Do NOT output words or explanations.
        """

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=3,
                do_sample=False,
                temperature=0.0
            )

qwen_debug_rows = []
def rerank_qwen(query, retrieved, tokenizer, model):
    
    import torch
    import re

    scored = []

    for doc, retr_score in retrieved:

        prompt = f"""Query: {query}

Document: {doc.page_content[:300]}

Score relevance from 0 to 10.

Output ONLY a number."""

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=3,
                do_sample=False,
                temperature=0.0
            )

        generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
        response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        match = re.search(r"\d+(\.\d+)?", response)

        if match:
            score = float(match.group())
        else:
            score = 0.0

        score = max(0.0, min(10.0, score))

        # ✅ STORE DEBUG ROW
        qwen_debug_rows.append({
            "query": query,
            "standard_code": doc.metadata.get("standard_code"),
            "raw_response": response,
            "parsed_score": score,
            "chunk_preview": doc.page_content[:200]
        })

        scored.append((doc, score))

    return sorted(scored, key=lambda x: x[1], reverse=True)

# def rerank_qwen(query, retrieved, tokenizer, model):

#     import torch
#     import re

#     scored = []

#     for doc, retr_score in retrieved:

#         prompt = f"""Query: {query}

# Document: {doc.page_content}

# Score relevance from 0 to 10.

# Output ONLY a number."""

#         inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#         with torch.no_grad():
#             output = model.generate(
#                 **inputs,
#                 max_new_tokens=3,
#                 do_sample=False,
#                 temperature=0.0
#             )

#         # ✅ ONLY decode generated part
#         generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
#         response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

#         print("RAW RESPONSE:", response)

#         match = re.search(r"\d+(\.\d+)?", response)

#         if match:
#             score = float(match.group())
#         else:
#             print("⚠️ NO NUMBER FOUND:", response)
#             score = 0.0

#         score = max(0.0, min(10.0, score))

#         scored.append((doc, score))

#     return sorted(scored, key=lambda x: x[1], reverse=True)
#         OLD PROMPT
        # prompt = f"""
        # You are a relevance ranking model.

        # Query:
        # {query}

        # Document:
        # {doc.page_content}

        # Is this document useful for answering the query?

        # Answer ONLY "yes" or "no".
        # """

#  NEW PROMPT
        # prompt = f"""You are a strict scoring funtion for medical policy documents.
        
        # Given a query and a document, assign a relevance score from 0 to 10.
        
        # 0 = completely irrelevant
        # 10 = perfectly answers the query 
        
        # consider:
        # - Does the document directly answer the query?
        # - Is it the correct policy standard and chapter?
        # - Is it complete or partial?
        
        # Query:
        # {query}
        
        # Document 
        # {doc.page_content}
        
        # Output ONLY a number between 0 to 10
        # Do NOT write words
        # Do NOT explain.
        # Do NOT include any text.
        # ONLY OUTPUT NUMBERS"""

       
# Prompt 1 scoring
        # score = 1.0 if "yes" in response else 0.0
# Prompt 2 scoring
        # try:
        #     score = float(response)
        # except:
        #     score = 0.0

        

# Metrics

In [11]:
# ================================
# METRICS (SHARED)
# ================================
def evaluate_metrics(docs_with_scores, expected_standard, threshold=0.5):

    docs = [d for d, _ in docs_with_scores]
    scores = [s for _, s in docs_with_scores]
    codes = [d.metadata.get("standard_code") for d in docs]

    # structural
    hit = expected_standard in codes[:5]

    # semantic
    answer_hit = any(
        (d.metadata.get("standard_code") == expected_standard and s > threshold)
        for d, s in docs_with_scores
    )

    # rank
    rank = None
    for i, c in enumerate(codes):
        if c == expected_standard:
            rank = i + 1
            break

    return hit, answer_hit, rank

        

# 9 Evaluation functions

In [12]:
import time

# ============================================================
# EVALUATION FUNCTION (REFACTORED)
# Change: Removed DB building from inside → function now ONLY evaluates
# ============================================================
# ===============================
# STORE TOP-K chunks to debug in excel
# ===============================
chunk_debug_rows = []
TOP_K_DEBUG_CHUNKS = 5
def store_chunks_DEBUG(mode_name, reranked_list, query, expected_standard, model_name):
    for rank, (doc, score) in enumerate(reranked_list[:TOP_K_DEBUG_CHUNKS], start=1):
        chunk_debug_rows.append({
            "model": model_name,
            "mode": mode_name,
            "query": query,
            "expected standard": expected_standard,
            
            "rank": rank,
            "Standard_code": doc.metadata.get("standard_code"),
            
            "is_correct_standard": doc.metadata.get("standard_code") == expected_standard,
            
            "score": score,
            "chunk_preview": doc.page_content[:300] #first 300 chars
            
        })
        
        
# def run_evaluation(dbs, gold_standard, model_name, k_values):

#     print(f"Evaluating: {model_name}")
#     print("="*50)

#     results = []

#     cross_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
#     tokenizer, qwen_model = load_qwen()

#     for item in gold_standard[:5]:
#         query = item["query"]
#         expected_standard = item["expected_standard"]

#         # ===============================
#         # RETRIEVE
#         # ===============================
#         retrieved = retrieve_all(query, dbs)

#         # ===============================
#         # BASELINE (NO RERANK)
#         # ===============================
        
#         base_scores = cross_model.predict([(query, doc.page_content) for doc, _ in retrieved])

#         baseline = [(doc, score) for (doc, _), score in zip(retrieved, base_scores)]

#         base_hit, base_answer_hit, base_rank = evaluate_metrics(
#             baseline, expected_standard
#         )

#         # ===============================
#         # CROSS RERANK
#         # ===============================
#         cross_reranked = rerank_cross(query, retrieved, cross_model)

#         cross_hit, cross_answer_hit, cross_rank = evaluate_metrics(
#             cross_reranked, expected_standard
#         )

#         # ===============================
#         # QWEN RERANK
#         # ===============================
#         qwen_reranked = rerank_qwen(query, retrieved, tokenizer, qwen_model)

#         qwen_hit, qwen_answer_hit, qwen_rank = evaluate_metrics(
#             qwen_reranked, expected_standard
#         )
        
#         store_chunks_DEBUG("baseline", baseline, query, expected_standard, model_name)
#         store_chunks_DEBUG("cross", cross_reranked, query, expected_standard, model_name)
#         store_chunks_DEBUG("qwen", qwen_reranked, query, expected_standard, model_name)
#         # ===============================
#         # STORE
#         # ===============================
#         row = {
#             "model": model_name,
#             "query": query,
#             "expected_standard": expected_standard,

#             "base_hit": base_hit,
#             "base_answer_hit": base_answer_hit,
#             "base_rank": base_rank,

#             "cross_hit": cross_hit,
#             "cross_answer_hit": cross_answer_hit,
#             "cross_rank": cross_rank,

#             "qwen_hit": qwen_hit,
#             "qwen_answer_hit": qwen_answer_hit,
#             "qwen_rank": qwen_rank,
#         }

#         results.append(row)

#     return results

def run_evaluation_clean(dbs, gold_standard):

    print("\n==============================")
    print("STARTING CLEAN DEBUG EVALUATION")
    print("==============================\n")

    results = []

    cross_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    tokenizer, qwen_model = load_qwen()

    for i, item in enumerate(gold_standard[:5], start=1):

        query = item["query"]
        expected = item["expected_standard"]

        print("\n===================================")
        print(f"QUERY {i}")
        print("===================================")
        print("Query:", query)
        print("Expected Standard:", expected)

        # =========================
        # STEP 1: RETRIEVE
        # =========================
        retrieved = retrieve_all(query, dbs)

        print("\n[RETRIEVAL]")
        print("Total retrieved:", len(retrieved))

        for j, (doc, score) in enumerate(retrieved[:5], start=1):
            print(f"Top {j}:",
                  doc.metadata.get("standard_code"),
                  "| Score:", round(score, 3))

        # =========================
        # STEP 2: BASELINE
        # =========================
        baseline = retrieved

        print("\n[BASELINE TOP 5]")
        for j, (doc, _) in enumerate(baseline[:5], start=1):
            print(f"{j}.", doc.metadata.get("standard_code"))

        # =========================
        # STEP 3: CROSS
        # =========================
        cross = rerank_cross(query, retrieved, cross_model)

        print("\n[CROSS RERANK TOP 5]")
        for j, (doc, score) in enumerate(cross[:5], start=1):
            print(f"{j}.", doc.metadata.get("standard_code"),
                  "| Score:", round(score, 3))

        # =========================
        # STEP 4: QWEN
        # =========================
        retrieved_for_qwen = retrieved[:5]

        print("\n[QWEN INPUT DOCS]")
        for j, (doc, _) in enumerate(retrieved_for_qwen, start=1):
            print(f"{j}.", doc.metadata.get("standard_code"))

        qwen = rerank_qwen(query, retrieved_for_qwen, tokenizer, qwen_model)

        print("\n[QWEN RERANK TOP 5]")
        for j, (doc, score) in enumerate(qwen[:5], start=1):
            print(f"{j}.", doc.metadata.get("standard_code"),
                  "| Score:", score)

        # =========================
        # STEP 5: EVALUATE
        # =========================
        base_hit, _, base_rank = evaluate_metrics(baseline, expected)
        cross_hit, _, cross_rank = evaluate_metrics(cross, expected)
        qwen_hit, _, qwen_rank = evaluate_metrics(qwen, expected)

        print("\n[EVALUATION]")
        print("Baseline → Hit:", base_hit, "| Rank:", base_rank)
        print("Cross    → Hit:", cross_hit, "| Rank:", cross_rank)
        print("Qwen     → Hit:", qwen_hit, "| Rank:", qwen_rank)

        results.append({
            "query": query,
            "expected": expected,

            "base_rank": base_rank,
            "cross_rank": cross_rank,
            "qwen_rank": qwen_rank,

            "base_hit": compute_hit(base_rank),
            "cross_hit": compute_hit(cross_rank),
            "qwen_hit": compute_hit(qwen_rank)
        })

    print("\n==============================")
    print("EVALUATION COMPLETE")
    print("==============================\n")

    return results

def compute_hit(rank):
    return rank is not None and rank <= 5

# Slice helper
def get_topk(codes, k_values):
    return {f"top{k}": codes[:k] for k in k_values}

# 10 Running evalutation

In [13]:
# ============================================================
# RUN EVALUATION (SIMPLIFIED DEBUG VERSION)
# ============================================================

all_results = []

start = time.time()

# results = run_evaluation(
#     dbs=dbs,  # already built earlier
#     gold_standard=gold_standard[:5],  # debug mode
#     model_name="BGE-large",
#     k_values=K_VALUES
# )

results = run_evaluation_clean(dbs, gold_standard)

all_results.extend(results)

duration = round(time.time() - start, 1)
print(f"\nTime taken: {duration}s")


STARTING CLEAN DEBUG EVALUATION



Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1292.59it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 398/398 [00:22<00:00, 17.86it/s, Materializing param=model.norm.weight]                               
Some parameters are on the meta device because they were offloaded to the disk.



QUERY 1
Query: What screening criteria should be used when a patient arrives at the emergency department?
Expected Standard: ACC.1

[RETRIEVAL]
Total retrieved: 10
Top 1: ACC.1 | Score: 0.445
Top 2: ACC.1 | Score: 0.509
Top 3: AOP.1.4 | Score: 0.582
Top 4: COP.3.5 | Score: 0.587
Top 5: ACC.2.3 | Score: 0.616

[BASELINE TOP 5]
1. ACC.1
2. ACC.1
3. AOP.1.4
4. COP.3.5
5. ACC.2.3


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



[CROSS RERANK TOP 5]
1. ACC.1 | Score: 5.082
2. ACC.1 | Score: 3.936
3. COP.3.5 | Score: 3.55
4. ACC.2.3 | Score: 3.522
5. ACC.2.3 | Score: 2.337

[QWEN INPUT DOCS]
1. ACC.1
2. ACC.1
3. AOP.1.4
4. COP.3.5
5. ACC.2.3

[QWEN RERANK TOP 5]
1. ACC.1 | Score: 0.0
2. ACC.1 | Score: 0.0
3. AOP.1.4 | Score: 0.0
4. COP.3.5 | Score: 0.0
5. ACC.2.3 | Score: 0.0

[EVALUATION]
Baseline → Hit: True | Rank: 1
Cross    → Hit: True | Rank: 1
Qwen     → Hit: True | Rank: 1

QUERY 2
Query: How should hospitals handle patients with communicable diseases during screening?
Expected Standard: ACC.1

[RETRIEVAL]
Total retrieved: 10
Top 1: ACC.1 | Score: 0.487
Top 2: PCI.12.2 | Score: 0.501
Top 3: PCI.12.2 | Score: 0.557
Top 4: SQE.8.3 | Score: 0.562
Top 5: PCI.12 | Score: 0.57

[BASELINE TOP 5]
1. ACC.1
2. PCI.12.2
3. PCI.12.2
4. SQE.8.3
5. PCI.12

[CROSS RERANK TOP 5]
1. ACC.1 | Score: 4.912
2. ACC.1 | Score: 4.415
3. PCI.12.1 | Score: 3.533
4. PCI.12 | Score: 3.441
5. PCI.12.2 | Score: 2.361

[QWEN INPUT D

In [ ]:
import pandas as pd

qwen_df = pd.DataFrame(qwen_debug_rows)

qwen_df.to_csv("QWEN_DEBUG_9.csv", index=False)

print("Saved: QWEN_DEBUG_9.csv")

df_all = pd.DataFrame(all_results)

df_all.to_csv("FINAL_RESULTS_9.csv", index=False)

print("Saved: FINAL_RESULTS_9.csv")

chunk_df = pd.DataFrame(chunk_debug_rows)
chunk_df.to_csv("Eval_Debug_9.csv", index = False)
print("Saved: Eval_Debug_9.csv")

Saved: QWEN_DEBUG_9.csv
Saved: FINAL_RESULTS_9.csv
Saved: Eval_CHUNK_Debug_9.csv


# 11 Summary table

In [ ]:
# ================================
# SUMMARY + EXCEL EXPORT (UPDATED)
# ================================

import pandas as pd

# ================================
# BUILD DATAFRAME
# ================================
df_all = pd.DataFrame(all_results)

# Sort for readability
df_all = df_all.sort_values(by=["model", "base_rank"])


# ================================
# SUMMARY TABLE
# ================================
summary_rows = []

for model_name in MODELS_TO_EVALUATE.keys():
    model_df = df_all[df_all["model"] == model_name]

    total = len(model_df)
    row = {"Model": model_name}

    for col in ["base", "cross", "qwen"]:

        # STRUCTURAL CORRECTNESS
        hit_count = model_df[f"{col}_hit"].sum()
        hit_rate = (hit_count / total * 100) if total > 0 else 0

        # SEMANTIC CORRECTNESS (VERY IMPORTANT)
        answer_count = model_df[f"{col}_answer_hit"].sum()
        answer_rate = (answer_count / total * 100) if total > 0 else 0

        # RANK (MRR style)
        def reciprocal(rank):
            return 1 / rank if rank else 0

        mrr = model_df[f"{col}_rank"].apply(reciprocal).mean()

        row[f"{col}_Hit"] = f"{hit_rate:.0f}% ({hit_count}/{total})"
        row[f"{col}_Answer"] = f"{answer_rate:.0f}% ({answer_count}/{total})"
        row[f"{col}_MRR"] = f"{mrr:.3f}"

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

print("\n=== FINAL SUMMARY ===")
print(summary_df.to_string(index=False))


# ================================
# SAVE SUMMARY CSV
# ================================
summary_df.to_csv("Eval_qwen_prompt1_SUMMARY_ALL_MODELS_7.csv", index=False)
print("\nSaved: Eval_qwen_prompt1_SUMMARY_ALL_MODELS_7.csv")


# ================================
# SAVE FULL RESULTS CSV
# ================================
df_save = df_all.copy()

# Clean booleans for readability
for col in df_save.columns:
    if "hit" in col:
        df_save[col] = df_save[col].astype(str)

df_save.to_csv("Eval_qwen_prompt1_FULL_RESULTS_7.csv", index=False)
print("Saved: Eval_qwen_prompt1_FULL_RESULTS_7.csv")

# ================================
# SAVE CHUNK DEBUG FILE
# ================================
chunk_df = pd.DataFrame(chunk_debug_rows)
chunk_df.to_csv("Eval_Debug_7.csv", index = False)
print("Saved: Eval_CHUNK_Debug_7.csv")


=== FINAL SUMMARY ===
            Model    base_Hit base_Answer base_MRR   cross_Hit cross_Answer cross_MRR    qwen_Hit qwen_Answer qwen_MRR
MiniLM (baseline) 88% (22/25) 80% (20/25)    0.870 80% (20/25)  80% (20/25)     0.920 88% (22/25)   0% (0/25)    0.870
        BGE-small 88% (22/25) 80% (20/25)    0.799 80% (20/25)  80% (20/25)     0.922 88% (22/25)   0% (0/25)    0.799
        BGE-large 84% (21/25) 84% (21/25)    0.922 80% (20/25)  84% (21/25)     0.865 84% (21/25)   0% (0/25)    0.922

Saved: Eval_qwen_prompt2_SUMMARY_ALL_MODELS.csv
Saved: Eval_qwen_prompt2_FULL_RESULTS.csv
Saved: Eval_CHUNK_DEBUG.csv
